# Notebook 22 ResNet50 Oracle

**ResNet-50 oracle (32x32 window) baseline.**

ResNet-50-oracle baseline for the clean-vs-noisy detection task.

Mirrors `raw_oracle` in exp07: takes a 32×32 window centered on the injected
patch, upscales to 224×224, and feeds into ImageNet-pretrained ResNet-50
(avgpool features, 2048-d). Trains the same logistic probe as exp07.

Runs all 5 perturbations (iso_blur p4 + p8, dir_blur_cranio p64,
reticular_fine p32, ground_glass p64) so we have the full grid.

Output rows are appended to exp07_<dataset>_rawpixel_baseline.parquet with
mode='resnet50_oracle'.

Usage:
    DATASET=nih python compute_resnet50_oracle.py


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory, vindr (where applicable)
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None when running gated models locally


In [ ]:
# Apply papermill parameters to environment
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms.functional as TF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

REPO_ROOT = Path(os.environ.get(
    "REPO_ROOT", "/home/saptpurk/embeddings-noise-eliminators"))
sys.path.insert(0, str(REPO_ROOT))

from common import (  # noqa: E402
    PARAMS,
    DirectionalMotionBlurInjector,
    GroundGlassInjector,
    LocalizedBlurInjector,
    ReticularPatternInjector,
    get_config,
    load_and_pad,
    load_disease_labels,
    stratified_split,
)

DATASET = os.environ.get("DATASET", "nih")
WORK = Path(os.environ.get(
    "OUTPUTS_DIR", "/home/saptpurk/embeddings-noise-eliminators/outputs"))
SEED = PARAMS.random_seed

# Same sample size as exp07.
N_TARGET = 20_000
ORACLE_CROP = 32        # match raw_oracle: 32x32 window around patch
RESNET_INPUT = 224
PERTURBATIONS = [
    ("iso_blur_p4", LocalizedBlurInjector(seed=SEED), 4),
    ("iso_blur_p8", LocalizedBlurInjector(seed=SEED), 8),
    ("dir_blur_cranio_p64",
     DirectionalMotionBlurInjector(
         seed=SEED, angle_deg=90.0,
         kernel_length=PARAMS.directional_kernel_length),
     64),
    ("reticular_fine_p32",
     ReticularPatternInjector(seed=SEED, period_px=3,
                              amplitude=PARAMS.reticular_amplitude),
     32),
    ("ground_glass_p64",
     GroundGlassInjector(seed=SEED, sigma_px=PARAMS.ground_glass_sigma,
                         amplitude=PARAMS.ground_glass_amplitude),
     64),
]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_resnet50():
    weights = tvm.ResNet50_Weights.IMAGENET1K_V2
    model = tvm.resnet50(weights=weights)
    model.fc = nn.Identity()
    model.eval().to(DEVICE)
    return model


def oracle_crop(img, patch_locations, crop=ORACLE_CROP):
    """Return a (crop, crop, 3) uint8 window centered on the artefact.

    Falls back to image center if no location given (matches raw_oracle).
    """
    if img.ndim == 2:
        img = np.repeat(img[:, :, None], 3, axis=2)
    elif img.shape[2] == 1:
        img = np.repeat(img, 3, axis=2)
    H, W = img.shape[:2]
    if not patch_locations:
        cy, cx = H // 2, W // 2
    else:
        loc = patch_locations[0]
        cy = loc["y"] + loc["size"] // 2
        cx = loc["x"] + loc["size"] // 2
    hh = crop // 2
    y0 = max(0, cy - hh)
    y1 = min(H, cy + hh)
    x0 = max(0, cx - hh)
    x1 = min(W, cx + hh)
    out = img[y0:y1, x0:x1]
    if out.shape[:2] != (crop, crop):
        # Pad to exact crop size if at the image border.
        pad_y = crop - out.shape[0]
        pad_x = crop - out.shape[1]
        out = np.pad(out, ((0, pad_y), (0, pad_x), (0, 0)),
                     mode="edge")
    return out


def preprocess_crop(crop):
    """crop is (32, 32, 3) uint8 → ResNet50 input tensor (3, 224, 224)."""
    t = torch.from_numpy(crop).permute(2, 0, 1).contiguous()
    t = TF.resize(t, [RESNET_INPUT, RESNET_INPUT], antialias=True)
    t = t.float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t - mean) / std


@torch.no_grad()
def extract_features(model, df, injector, patch_size, target_size,
                     batch_size=64):
    feats_clean, feats_pert = [], []
    bc, bp = [], []

    def _flush():
        if bc:
            xc = torch.stack(bc).to(DEVICE, non_blocking=True)
            xp = torch.stack(bp).to(DEVICE, non_blocking=True)
            feats_clean.append(model(xc).cpu().numpy())
            feats_pert.append(model(xp).cpu().numpy())
            bc.clear()
            bp.clear()

    for _, row in tqdm(df.iterrows(), total=len(df),
                       desc=f"resnet50_oracle {injector.__class__.__name__}/{patch_size}"):
        clean = load_and_pad(row["image_path"], target_size)
        noisy, meta = injector(clean, patch_size=patch_size, num_patches=1,
                               image_path=row["image_path"])
        locs = meta["patch_locations"]
        bc.append(preprocess_crop(oracle_crop(clean, locs)))
        bp.append(preprocess_crop(oracle_crop(noisy, locs)))
        if len(bc) >= batch_size:
            _flush()
    _flush()
    return np.vstack(feats_clean), np.vstack(feats_pert)


def fit_eval(X_train, y_train, X_test, y_test, n_boot=PARAMS.n_bootstrap):
    scaler = StandardScaler().fit(X_train)
    Xtr = scaler.transform(X_train)
    Xte = scaler.transform(X_test)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_C, best_cv = None, -np.inf
    for C in PARAMS.lr_C_grid:
        cv = []
        for tr_idx, va_idx in skf.split(Xtr, y_train):
            clf = LogisticRegression(C=C, max_iter=PARAMS.lr_max_iter,
                                     class_weight="balanced",
                                     solver="lbfgs", random_state=SEED)
            clf.fit(Xtr[tr_idx], y_train[tr_idx])
            cv.append(roc_auc_score(
                y_train[va_idx],
                clf.predict_proba(Xtr[va_idx])[:, 1]))
        m = float(np.mean(cv))
        if m > best_cv:
            best_cv, best_C = m, C
    clf = LogisticRegression(C=best_C, max_iter=PARAMS.lr_max_iter,
                             class_weight="balanced",
                             solver="lbfgs", random_state=SEED)
    clf.fit(Xtr, y_train)
    proba = clf.predict_proba(Xte)[:, 1]
    auc = float(roc_auc_score(y_test, proba))
    p, r, t = precision_recall_curve(y_test, proba)
    if len(t):
        f1s = 2 * p[1:] * r[1:] / (p[1:] + r[1:] + 1e-12)
        thr = float(t[int(np.argmax(f1s))])
    else:
        thr = 0.5
    f1 = float(f1_score(y_test, (proba >= thr).astype(int), zero_division=0))
    rng = np.random.default_rng(SEED)
    n = len(y_test)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_test[idx])) > 1:
            aucs.append(roc_auc_score(y_test[idx], proba[idx]))
    ci = (float(np.percentile(aucs, 2.5)) if aucs else auc,
          float(np.percentile(aucs, 97.5)) if aucs else auc)
    return dict(auc=auc, auc_ci_low=ci[0], auc_ci_high=ci[1],
                f1=f1, threshold=thr, best_C=float(best_C))


CFG = get_config()
print(f"Dataset: {CFG.name} | Device: {DEVICE} | Oracle crop: {ORACLE_CROP}")

df_all = load_disease_labels(CFG, ["cardiomegaly"])
rng = np.random.default_rng(SEED)
n = min(N_TARGET, len(df_all))
idx = rng.choice(len(df_all), size=n, replace=False)
df = df_all.iloc[idx].reset_index(drop=True)
df["_stratum"] = "0"
train_df, test_df = stratified_split(df, "_stratum", test_frac=0.2, seed=SEED)
print(f"train={len(train_df)}  test={len(test_df)}")

model = build_resnet50()

records = []
for pert_name, injector, patch_size in PERTURBATIONS:
    print(f"\n--- {pert_name} ---")
    Xc_tr, Xp_tr = extract_features(model, train_df, injector,
                                    patch_size, CFG.target_size)
    Xc_te, Xp_te = extract_features(model, test_df, injector,
                                    patch_size, CFG.target_size)
    Xtr = np.vstack([Xc_tr, Xp_tr])
    ytr = np.concatenate([np.zeros(len(Xc_tr)),
                          np.ones(len(Xp_tr))]).astype(int)
    Xte = np.vstack([Xc_te, Xp_te])
    yte = np.concatenate([np.zeros(len(Xc_te)),
                          np.ones(len(Xp_te))]).astype(int)
    res = fit_eval(Xtr, ytr, Xte, yte)
    print(f"  resnet50_oracle  AUC={res['auc']:.4f} "
          f"[{res['auc_ci_low']:.4f}, {res['auc_ci_high']:.4f}]")
    records.append(dict(
        dataset=CFG.dataset, perturbation=pert_name, mode="resnet50_oracle",
        auc=res["auc"], auc_ci_low=res["auc_ci_low"],
        auc_ci_high=res["auc_ci_high"], f1=res["f1"],
        threshold=res["threshold"], best_C=res["best_C"],
        n_features=int(Xtr.shape[1]),
        n_train=int(len(ytr)), n_test=int(len(yte)),
    ))

new_rows = pd.DataFrame(records)
exp07_dir = WORK / f"v4_exp07_rawpixel_baseline_{CFG.dataset}"
exp07_dir.mkdir(parents=True, exist_ok=True)
existing = exp07_dir / f"exp07_{CFG.dataset}_rawpixel_baseline.parquet"
if existing.exists():
    df_existing = pd.read_parquet(existing)
    drop_mask = ((df_existing["mode"] == "resnet50_oracle")
                 & (df_existing["perturbation"].isin(
                     [r["perturbation"] for r in records])))
    df_existing = df_existing[~drop_mask]
    merged = pd.concat([df_existing, new_rows], ignore_index=True)
else:
    merged = new_rows
merged.to_parquet(existing, index=False)
print(f"\nWrote {len(new_rows)} new rows -> {existing}")
print(merged[merged["mode"].isin(["resnet50", "resnet50_oracle"])][
    ["perturbation", "mode", "auc", "auc_ci_low", "auc_ci_high"]].to_string())
